# Lineage ledger, sidecar annotations and dataset metadata (FAIR)

Builds, for the images of the final dataset:

1. **the manipulation ledger**, `fake/manipulations_extraction.csv`: one row per generation event, append-only (`LINEAGE.md`);
2. **one JSON sidecar per image**, `fake/by_method/<method>/<images>/<sample_id>.json`, validated against `schemas/manipulation_schema.json` **before** it is written (`ANNOTATIONS.md`);
3. **dataset-level metadata**: `datapackage.json` (title, creators, licence, sources, resources), a machine-readable description of the ledger columns (`fake/manipulations_extraction.schema.json`) and a copy of the sidecar schema, so the output folder is self-describing;
4. **a FAIR check** that lists what is still missing, principle by principle.

Inputs: the generation log (`data.jsonl`: prompt, source frame, mask, seed), `kept.csv` (the images that survived the filter) and, optionally, the origin manifest of the source collection.

The schema was written for videos and the original data has no licence, creators or DOI: everything the notebook cannot know is a constant in the next cell (**read and fill it before publishing**). `WRITE = False` only previews. Nothing here replaces a rights check on the source frames.

In [ ]:
from pathlib import Path

# ---- inputs ------------------------------------------------------------------------------------------------------
GEN_LOG  = Path("../datasets/archive-meta/results.jsonl")                        # generation log: record_id, prompt, image_path, mask_path, result[url, seed]
KEPT     = Path("../datasets/archive-meta/kept.csv")         # the images of the final dataset (column `file` = generated image name)
INPAINTS = Path("/Scratch/Shullani/Dataset/fooocus-cinematic/inpaints")   # folder with the generated images (file names as in kept.csv)
SCHEMA   = Path("../datasets/archive-meta/manipulation_schema.json")   # the schema every sidecar is validated against
ORIGIN_MANIFEST = Path("../datasets/archive-meta/original_manifest.csv")      # optional: manifest of the source collection (lang, video, frame, timestamp_s, person_score, bbox_*)
SOURCE_FRAMES_DIR = None                             # optional: local folder that contains `extracted_frames/` -> the source frames are hashed (frame_sha256)
PARENT_SIDECAR_UUIDS = None    # optional csv `file,uuid`: DARDcollect SIDECAR uuid of each source frame (-> parent.uuid)
PARENT_LEDGER_UUIDS  = None    # optional csv `file,uuid`: DARDcollect extraction-ROW uuid of each source frame (-> ledger parent_uuid)

# ---- output ------------------------------------------------------------------------------------------------------
OUT_ROOT   = Path("lineage_out")                     # the output root: paths in the ledger are relative to it
MEDIA_MODE = "symlink"                               # how the image is placed beside its sidecar as <sample_id>.<ext>: symlink | copy | reference
WRITE      = False                                   # False = preview only; True = write ledger, sidecars, media and dataset metadata

# ---- the manipulation --------------------------------------------------------------------------------------------
METHOD            = "fooocus_inpaint"                # concrete generator, goes in the ledger `method` column and in the folder name
MANIPULATION_TYPE = "inpainting"                     # semantic label (ground truth for a detector). Not in the original enum of the schema:
                                                     # "inpainting" was added to schemas/manipulation_schema.json, add it to the upstream schema too
ANNOTATOR         = "eval_suite/export_lineage.ipynb"        # the script/notebook that writes the sidecars: a path relative to REPO_ROOT, must exist
REPO_ROOT         = Path("..")                               # root of this repository, seen from the notebook folder
SOURCE_TYPE       = "frame"                          # ledger `source_type`; the docs only list `face_crop` (depth 0) and `generated`
MEDIA_DIR         = "images"                         # folder name where the docs say `videos`
GENERATOR         = {"model": "fooocus", "version": None, "repo": None, "weights": None}   # identity of the generator (None = omit): the generator
                                                     # lives in a separate repo, put its URL in `repo` and its commit/tag in `version`
ANNOTATED_AT      = "file_mtime"                     # annotated_at = modification time of the generated image ("file_mtime") or "now"

# ---- origin of the source frames ---------------------------------------------------------------------------------
SOURCE_COLLECTION = "DARD"                           # name of the collection the source frames come from
SOURCE_MARKER     = "extracted_frames"               # source paths are made portable: everything before this folder name is dropped

# ---- dataset-level metadata (Findable / Reusable): FILL IN BEFORE PUBLISHING -------------------------------------------
DATASET = {
    "name": "archive-face-inpainting",               # Data Package name: lowercase letters, digits, - _ .
    "title": "Face inpainting of archive-film frames (Fooocus)",
    "description": "Frames from archive films in which a person's face region was regenerated with a text-prompted "
                   "inpainting model, with the mask, prompt and provenance of every image.",
    "version": "1.0.0",
    "creators": ["University of Florence, LESC"],                                  # e.g. ["Name Surname (Institution)", ...]; empty = not declared (flagged)
    "license": None,                                 # SPDX id, e.g. "CC-BY-NC-4.0"; None = NOT declared (flagged). The repo's GPL-3.0 covers code, not data
    "rights_note": "",                               # rights of the source frames (films) and of the people shown: say what you checked
    "doi": None,                                     # persistent identifier, once the dataset is deposited (e.g. "10.5281/zenodo.1234567")
    "keywords": ["inpainting", "face", "manipulation detection", "archive film"],
}

In [ ]:
import csv, hashlib, json, os, re, shutil, uuid
from datetime import datetime, timezone
from urllib.parse import urlparse

import pandas as pd
from jsonschema import Draft7Validator, FormatChecker

FAKE = OUT_ROOT / "fake"
LEDGER = FAKE / "manipulations_extraction.csv"
LEDGER_SCHEMA = FAKE / "manipulations_extraction.schema.json"
SAMPLE_INDEX = FAKE / "sample_index.csv"             # stable sample_id <-> record_id, so ids survive a change of the kept set
UUID_RE = "^[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$"

# the ledger columns: (name, Table Schema type, description, constraints). The list is also the canonical header.
LEDGER_FIELDS = [
    ("uuid", "string", "Identity of this ledger ROW (UUID v4), computed at write time. Independent of the sidecar uuid.",
     {"required": True, "unique": True, "pattern": UUID_RE}),
    ("parent_uuid", "string", "Ledger-row uuid of the direct input (a DARDcollect extraction row at depth 0, the parent manipulation's row when chained). Empty when not known.",
     {"pattern": UUID_RE}),
    ("timestamp", "datetime", "UTC time the row was written, computed at write time.", {"required": True}),
    ("manipulation_id", "string", "Sample id (s0000, s0001, ...). Joins the row to its sidecar <manipulation_id>.json.", {"required": True}),
    ("method", "string", "Concrete generator that produced the image; the sidecar manipulation_type is derived from it.", {"required": True}),
    ("source_type", "string", "What the input is: 'frame' (a full source frame), 'face_crop', or 'generated' (a prior manipulation).", {}),
    ("source_path", "string", "Portable path of the input, relative to the source collection root (starts at extracted_frames/...). Never absolute.", {}),
    ("cross_identity", "boolean", "Decision scalar for videos: the visible face differs from the clip that supplied the frames. Empty = not applicable or unknown (still images).", {}),
    ("frames", "integer", "Frames of the produced video. Empty for still images.", {}),
    ("duration_seconds", "number", "Duration of the produced video. Empty for still images.", {}),
    ("sha256", "string", "SHA-256 of the produced file. With manipulation_id it is the dedup key of the ledger.",
     {"required": True, "pattern": "^[0-9a-f]{64}$"}),
    ("output_path", "string", "Path of the produced media, relative to the output root.", {"required": True}),
]
FIELDNAMES = [f[0] for f in LEDGER_FIELDS]

# shared JSON-LD block, copied from s0000.example.json unchanged
CONTEXT = {"dct": "http://purl.org/dc/terms/", "prov": "http://www.w3.org/ns/prov#", "title": "dct:title",
           "creator": "dct:creator", "date": "dct:date", "license": "dct:license", "uuid": "dct:identifier",
           "schema_version": "dct:conformsTo", "parent": "prov:wasDerivedFrom"}

schema = json.loads(SCHEMA.read_text())
Draft7Validator.check_schema(schema)
validator = Draft7Validator(schema, format_checker=FormatChecker())


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def utc(ts=None):
    return (datetime.fromtimestamp(ts, timezone.utc) if ts else datetime.now(timezone.utc)).isoformat()


def read_map(path):
    """csv `file,uuid` -> {file basename: uuid}"""
    if not path:
        return {}
    df = pd.read_csv(path, dtype=str)
    return {Path(f).name: u for f, u in zip(df["file"], df["uuid"])}


class ManipulationLogger:
    """Append-only ledger. uuid and timestamp are computed here, never by the caller. The header is validated on
    read: a file whose columns differ from FIELDNAMES raises instead of being misread. Re-running over unchanged
    media is a no-op (dedup on (manipulation_id, sha256)); a regenerated image has other bytes, so it appends a row."""

    def __init__(self, path):
        self.path = Path(path)
        self.rows = []
        if self.path.exists():
            with open(self.path, newline="", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames != FIELDNAMES:
                    raise ValueError(f"{self.path}: header {reader.fieldnames} differs from the canonical {FIELDNAMES}")
                self.rows = list(reader)
        self.keys = {(r["manipulation_id"], r["sha256"]) for r in self.rows}

    def append(self, **fields):
        row = {k: fields.get(k, "") for k in FIELDNAMES}
        row["uuid"], row["timestamp"] = str(uuid.uuid4()), utc()
        if (row["manipulation_id"], row["sha256"]) in self.keys:
            return None
        new_file = not self.path.exists()
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.path, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=FIELDNAMES)
            if new_file:
                w.writeheader()
            w.writerow(row)
        self.rows.append(row)
        self.keys.add((row["manipulation_id"], row["sha256"]))
        return row


def sample_ids(record_ids):
    """Stable sample ids s0000, s0001, ... assigned in order of first appearance and remembered on disk."""
    index = {}
    if SAMPLE_INDEX.exists():
        index = dict(pd.read_csv(SAMPLE_INDEX, dtype=str)[["record_id", "sample_id"]].values)
    new = [r for r in record_ids if r not in index]
    for r in new:
        index[r] = f"s{len(index):04d}"
    return index, new


# ---- origin of the source frames -----------------------------------------------------------------------------------
SRC_RE = re.compile(rf"{re.escape(SOURCE_MARKER)}/(?P<lang>[^/]+)/(?P<video>[^/]+)/(?P<frame>frame_\d+)\.\w+$")
_frame_hashes = {}


def parse_source(image_path):
    """.../extracted_frames/<lang>/<video>/<frame>.png -> lang, video, frame and the PORTABLE path (from SOURCE_MARKER on)."""
    m = SRC_RE.search(str(image_path).replace("\\", "/"))
    if not m:
        return None
    d = m.groupdict()
    d["path"] = f"{SOURCE_MARKER}/{d['lang']}/{d['video']}/{d['frame']}{Path(str(image_path)).suffix}"
    return d


def load_manifest(path):
    """origin manifest -> {(lang, video, frame): row}"""
    if not path or not Path(path).exists():
        return {}
    m = pd.read_csv(path, dtype={"frame": str, "lang": str, "video": str})
    return {(r["lang"], r["video"], r["frame"]): r for r in m.to_dict("records")}


def build_origin(rec, manifest, with_hash):
    s = parse_source(rec["image_path"])
    if not s:
        return None
    o = {"collection": SOURCE_COLLECTION, "lang": s["lang"], "video": s["video"], "frame": s["frame"], "path": s["path"]}
    row = manifest.get((s["lang"], s["video"], s["frame"]))
    if row:
        o.update(manifest_idx=int(row["idx"]), timestamp_s=float(row["timestamp_s"]), person_score=float(row["person_score"]),
                 face_visible=str(row["face_visible"]).strip().lower() == "true",
                 person_bbox_xyxy=[float(row[k]) for k in ("bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2")])
    if with_hash and SOURCE_FRAMES_DIR:
        p = Path(SOURCE_FRAMES_DIR) / s["path"]
        if p.exists():
            if p not in _frame_hashes:
                _frame_hashes[p] = sha256_of(p)
            o["frame_sha256"] = _frame_hashes[p]
    return o


def build_sidecar(sample_id, rec, size, output_file, annotated_at, own_uuid, parent_uuid, origin):
    frame = Path(rec["image_path"]).name
    parent = {"file": frame, "sample_id": rec["record_id"]}
    face = {"file": frame, "sample_id": rec["record_id"], "role": "face"}
    if parent_uuid:                                    # the real FAIR identifier, when DARDcollect sidecars are known
        parent["uuid"] = face["uuid"] = parent_uuid
    generator = {"seed": int(rec["seed"]) if str(rec["seed"]).isdigit() else rec["seed"],
                 "params": {"prompt": rec["prompt"], "mask_file": Path(rec["mask_path"]).name,
                            "record_id": rec["record_id"], "generated_file": rec["file"]}}
    generator.update({k: v for k, v in GENERATOR.items() if v})
    what = f"{origin['video']} / {origin['frame']}" if origin else frame
    sidecar = {"@context": CONTEXT, "uuid": own_uuid, "schema_version": "1.0", "manipulation_type": MANIPULATION_TYPE,
               "title": f"{rec['record_id']}: {METHOD} of {what}", "date": annotated_at[:10],
               "annotator": ANNOTATOR, "annotated_at": annotated_at, "output_file": output_file,
               "parent": parent, "manipulation_depth": 0,
               "sources": {"identity": face, "audio": {"role": "generated"}, "motion": dict(face)},
               "generator": generator, "video_info": {"width": int(size[0]), "height": int(size[1])}}
    if DATASET.get("creators"):
        sidecar["creator"] = list(DATASET["creators"])
    if DATASET.get("license"):
        sidecar["license"] = DATASET["license"]
    if origin:
        sidecar["origin"] = origin
    return sidecar


def check(sidecar):
    errors = sorted(validator.iter_errors(sidecar), key=lambda e: list(e.path))
    return [f"{'/'.join(map(str, e.path)) or '<root>'}: {e.message}" for e in errors]

## 1. Join the generation log with the kept images and the origin manifest

A generated image is `<record_id>_inpainted-0.png`; its name comes from the `url` in the log and is the `file` of `kept.csv`. The source frame of every record is read from `image_path` (`.../extracted_frames/<lang>/<video>/<frame>.png`); the origin manifest adds timestamp and person detection where the frame is listed there.

In [ ]:
log = pd.read_json(GEN_LOG, lines=True, dtype=False)
log["file"] = log["result"].map(lambda r: Path(urlparse(r[0]["url"]).path).name)
log["seed"] = log["result"].map(lambda r: r[0]["seed"])
log["finish"] = log["result"].map(lambda r: r[0].get("finish_reason"))

kept = pd.read_csv(KEPT)
df = kept[["file", "width", "height"]].merge(log[["record_id", "prompt", "image_path", "mask_path", "seed", "file", "finish"]],
                                             on="file", how="left")
missing = df[df["record_id"].isna()]
print(f"{len(kept)} kept images | {len(df) - len(missing)} found in the generation log | {len(missing)} not found")
if len(missing):
    print("  not in the log, skipped:", missing["file"].head(5).tolist())
df = df[df["record_id"].notna()].reset_index(drop=True)
print("finish_reason:", df["finish"].value_counts().to_dict(), "| distinct seeds:", df["seed"].nunique())

MANIFEST = load_manifest(ORIGIN_MANIFEST)
keys = [parse_source(p) for p in df["image_path"]]
parsed = sum(k is not None for k in keys)
in_manifest = sum(k is not None and (k["lang"], k["video"], k["frame"]) in MANIFEST for k in keys)
print(f"source frame parsed from image_path: {parsed}/{len(df)} | frames of {len({k['video'] for k in keys if k})} videos")
print(f"origin manifest: {len(MANIFEST)} rows | records whose frame is in the manifest: {in_manifest}/{len(df)}")
if MANIFEST and in_manifest < len(df):
    print("  [info] the manifest does not list most of these frames: only the frames it lists get timestamp / person fields")
df.head(3)

## 2. Preview one record

In [ ]:
SOURCE_UUIDS, LEDGER_UUIDS = read_map(PARENT_SIDECAR_UUIDS), read_map(PARENT_LEDGER_UUIDS)
rec = df.iloc[0].to_dict()
frame = Path(rec["image_path"]).name
preview = build_sidecar("s0000", rec, (rec["width"], rec["height"]), "s0000" + Path(rec["file"]).suffix,
                        utc(), str(uuid.uuid4()), SOURCE_UUIDS.get(frame), build_origin(rec, MANIFEST, with_hash=False))
print(json.dumps(preview, indent=2, ensure_ascii=False))
problems = check(preview)
print("\nschema:", "valid" if not problems else problems)

## 3. Write

Per image: sha256 of the file, the media placed as `<sample_id>.<ext>`, the sidecar (validated first; a sidecar that violates the schema is never written; an existing sidecar keeps its `uuid` so chained children never dangle) and then the ledger row. Source paths are written in their **portable** form (no machine-specific prefix).

In [ ]:
if MANIPULATION_TYPE is None:
    raise SystemExit("Set MANIPULATION_TYPE in the configuration cell (see the options there) before writing.")
if not (REPO_ROOT / ANNOTATOR).exists():
    raise SystemExit(f"ANNOTATOR {ANNOTATOR!r} does not exist under REPO_ROOT={REPO_ROOT.resolve()}: the schema requires it to name a real file.")

index, new_ids = sample_ids(df["record_id"].tolist())
media_dir = FAKE / "by_method" / METHOD / MEDIA_DIR
logger = ManipulationLogger(LEDGER)
rows_before = len(logger.rows)
stats = dict(written=0, unchanged=0, no_file=0, invalid=0)
invalid, ORIGINS = [], {}

for rec in df.to_dict("records"):
    src = INPAINTS / rec["file"]
    if not src.exists():
        stats["no_file"] += 1
        continue
    sid = index[rec["record_id"]]
    out_file = sid + src.suffix
    sidecar_path = media_dir / f"{sid}.json"
    prev = json.loads(sidecar_path.read_text()) if sidecar_path.exists() else None
    own_uuid = prev["uuid"] if prev else str(uuid.uuid4())
    annotated = utc(src.stat().st_mtime) if ANNOTATED_AT == "file_mtime" else utc()
    frame = Path(rec["image_path"]).name
    origin = ORIGINS[rec["record_id"]] = build_origin(rec, MANIFEST, with_hash=WRITE)
    sidecar = build_sidecar(sid, rec, (rec["width"], rec["height"]), out_file if MEDIA_MODE != "reference" else src.name,
                            annotated, own_uuid, SOURCE_UUIDS.get(frame), origin)
    errors = check(sidecar)
    if errors:
        stats["invalid"] += 1
        invalid.append((rec["file"], errors))
        continue
    if not WRITE:
        stats["written"] += 1
        continue

    sha = sha256_of(src)
    if MEDIA_MODE == "reference":
        output_path = str(src.resolve())
    else:
        media_dir.mkdir(parents=True, exist_ok=True)
        dst = media_dir / out_file
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        shutil.copy2(src, dst) if MEDIA_MODE == "copy" else os.symlink(src.resolve(), dst)
        output_path = dst.relative_to(OUT_ROOT).as_posix()
    media_dir.mkdir(parents=True, exist_ok=True)
    sidecar_path.write_text(json.dumps(sidecar, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    row = logger.append(parent_uuid=LEDGER_UUIDS.get(frame, ""), manipulation_id=sid, method=METHOD,
                        source_type=SOURCE_TYPE, source_path=origin["path"] if origin else frame, cross_identity="",
                        frames="", duration_seconds="", sha256=sha, output_path=output_path)
    stats["written" if row else "unchanged"] += 1

if WRITE and new_ids:
    SAMPLE_INDEX.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(sorted(index.items(), key=lambda kv: kv[1]), columns=["record_id", "sample_id"]).to_csv(SAMPLE_INDEX, index=False)

print(("WRITE" if WRITE else "PREVIEW (WRITE = False, nothing written)"), stats)
print(f"ledger rows: {rows_before} -> {len(logger.rows)}")
for f, e in invalid[:5]:
    print("  invalid:", f, e)

## 4. Dataset-level metadata

Writes, in the output root, what makes the folder self-describing: the description of the ledger columns (a Frictionless *Table Schema*), a copy of the sidecar schema, and `datapackage.json` (title, description, version, creators, licence, the source films, and the resources: ledger and sidecars with their schemas and the ledger hash). Re-run after any change to the ledger: the hash is recomputed.

In [ ]:
def write_package():
    fields = [{"name": n, "type": t, "description": d, **({"constraints": c} if c else {})} for n, t, d, c in LEDGER_FIELDS]
    LEDGER_SCHEMA.write_text(json.dumps({"fields": fields, "primaryKey": ["uuid"], "missingValues": [""],
                                         "description": "Append-only ledger: one row per generation event. (manipulation_id, sha256) is also unique."},
                                        indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    (OUT_ROOT / "schemas").mkdir(parents=True, exist_ok=True)
    shutil.copy2(SCHEMA, OUT_ROOT / "schemas" / "manipulation_schema.json")

    sidecar_paths = sorted(p.relative_to(OUT_ROOT).as_posix() for p in (FAKE / "by_method").rglob("*.json"))
    videos = {}
    for o in ORIGINS.values():
        if o:
            videos.setdefault((o["lang"], o["video"]), set()).add(o["frame"])
    pkg = {"$schema": "https://datapackage.org/profiles/2.0/datapackage.json",
           "name": DATASET["name"], "title": DATASET["title"], "description": DATASET["description"],
           "version": DATASET["version"], "created": utc(), "keywords": DATASET["keywords"],
           "resources": [
               {"name": "manipulations-extraction", "path": LEDGER.relative_to(OUT_ROOT).as_posix(), "format": "csv",
                "mediatype": "text/csv", "encoding": "utf-8", "hash": "sha256:" + sha256_of(LEDGER),
                "schema": LEDGER_SCHEMA.relative_to(OUT_ROOT).as_posix(),
                "description": "Manipulation ledger: one row per generation event (append-only)."},
               {"name": "sidecars", "path": sidecar_paths, "format": "json", "mediatype": "application/json",
                "description": f"One JSON annotation per image ({len(sidecar_paths)}), each validated against the JSON Schema in the "
                               "'manipulation-schema' resource. Each sits beside its image <sample_id>.<ext>; the image is named by the "
                               "same sample id as the ledger's manipulation_id."},
               {"name": "manipulation-schema", "path": "schemas/manipulation_schema.json", "type": "file", "format": "json",
                "mediatype": "application/schema+json", "description": "JSON Schema (draft-07) of the sidecars."}],
           "sources": [{"title": f"{v} ({l})", "description": f"{SOURCE_COLLECTION} source film, {len(fr)} frames used"}
                       for (l, v), fr in sorted(videos.items())]}
    if DATASET.get("doi"):
        pkg["id"] = "https://doi.org/" + DATASET["doi"]
    if DATASET.get("license"):
        pkg["licenses"] = [{"name": DATASET["license"]}]
    if DATASET.get("creators"):
        pkg["contributors"] = [{"title": c, "roles": ["author"]} for c in DATASET["creators"]]
    if DATASET.get("rights_note"):
        pkg["description"] += "\n\nRights: " + DATASET["rights_note"]
    (OUT_ROOT / "datapackage.json").write_text(json.dumps(pkg, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return pkg

if WRITE:
    pkg = write_package()
    print("wrote datapackage.json,", LEDGER_SCHEMA.name, "and schemas/manipulation_schema.json")
    print(f"  {len(pkg['resources'][1]['path'])} sidecars, {len(pkg['sources'])} source films, licence: {DATASET.get('license')}, creators: {len(DATASET.get('creators') or [])}")
    try:                                               # optional: not a dependency of the project (pip install frictionless)
        from frictionless import validate as fl_validate
        rep_ = fl_validate(str(OUT_ROOT / "datapackage.json"))
        print("frictionless validation of datapackage.json and the ledger:", "valid" if rep_.valid else [t.errors for t in rep_.tasks if not t.valid])
    except ImportError:
        print("(frictionless not installed: datapackage.json was not validated; pip install frictionless to check it)")
else:
    print("WRITE = False: dataset metadata not written")

## 5. Verify

Everything is re-read from disk: every ledger row has a sidecar with the same `manipulation_id`, every sidecar validates, the `sha256` of the **current** row matches the file, and the two id spaces never overlap.

In [ ]:
led = pd.read_csv(LEDGER, dtype=str, keep_default_na=False) if LEDGER.exists() else pd.DataFrame(columns=FIELDNAMES)
sidecars = {p.stem: json.loads(p.read_text()) for p in (FAKE / "by_method").rglob("*.json")} if FAKE.exists() else {}
current = led.drop_duplicates("manipulation_id", keep="last")             # the current row of a sample = the newest
report = {
    "ledger rows": len(led), "samples in the ledger": led["manipulation_id"].nunique(), "sidecars": len(sidecars),
    "ledger without sidecar": sorted(set(led["manipulation_id"]) - set(sidecars))[:5],
    "sidecar without ledger": sorted(set(sidecars) - set(led["manipulation_id"]))[:5],
    "sidecars failing the schema": sum(bool(check(s)) for s in sidecars.values()),
    "row uuid == sidecar uuid (must be 0)": len(set(led["uuid"]) & {s["uuid"] for s in sidecars.values()}),
}
def sha_ok(r):
    p = OUT_ROOT / r["output_path"] if not Path(r["output_path"]).is_absolute() else Path(r["output_path"])
    return p.exists() and sha256_of(p) == r["sha256"]
report["current rows whose sha256 matches the file"] = f"{int(current.apply(sha_ok, axis=1).sum())}/{len(current)}" if len(current) else "0/0"
for k, v in report.items():
    print(f"{k}: {v}")

## 6. FAIR check

What the written files satisfy and what is still missing, by principle. `OK` = met, `WARN` = met only partly or depends on a step outside this notebook (for example depositing the dataset), `MISSING` = not met. The list is the to-do for publishing.

In [ ]:
def abs_strings(obj):
    """strings that look like absolute paths anywhere in a JSON-like object"""
    if isinstance(obj, dict):
        for v in obj.values():
            yield from abs_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from abs_strings(v)
    elif isinstance(obj, str) and (obj.startswith("/") or re.match(r"^[A-Za-z]:[\\/]", obj)):
        yield obj


def fair_report():
    R, n = [], max(len(sidecars), 1)
    pct = lambda k: f"{100 * k / n:.0f}% ({k}/{len(sidecars)})"
    add = lambda p, c, s, d="": R.append({"principle": p, "check": c, "status": s, "detail": d})
    sc = list(sidecars.values())
    uuids = [s["uuid"] for s in sc]
    add("F1", "unique identifier for every sidecar and ledger row", "OK" if len(set(uuids)) == len(uuids) and led["uuid"].is_unique and len(sc) else "MISSING", f"{len(uuids)} sidecar uuids")
    add("F1", "persistent identifier for the dataset (DOI / handle)", "OK" if DATASET.get("doi") else "WARN",
        DATASET.get("doi") or "the uuids are local: deposit the dataset and set DATASET['doi']")
    for k in ("title", "creator", "date", "license"):
        c = sum(k in s for s in sc)
        add("F2", f"sidecars carry `{k}`", "OK" if c == len(sc) and sc else "MISSING", pct(c))
    add("F3", "metadata name the data they describe (output_file, sha256 in the ledger)",
        "OK" if len(sc) and all("output_file" in s for s in sc) and led["sha256"].str.len().eq(64).all() else "MISSING", "")
    add("F4", "dataset-level record (datapackage.json)", "OK" if (OUT_ROOT / "datapackage.json").exists() else "MISSING", "")
    absn = sum(1 for s in sc for _ in abs_strings(s)) + int(led["source_path"].str.startswith("/").sum() + led["output_path"].str.startswith("/").sum())
    add("A1", "no machine-specific absolute paths in sidecars and ledger", "OK" if absn == 0 else "WARN", f"{absn} absolute path(s)" if absn else "")
    add("A1", "retrievable through a standard protocol", "WARN",
        "DOI declared: check that it resolves to the deposited files" if DATASET.get("doi") else "deposit the dataset in a repository (e.g. Zenodo) and set DATASET['doi']")
    add("A2", "metadata kept apart from the data (sidecars, ledger)", "OK" if len(sc) else "MISSING", "metadata stay usable if images are withdrawn")
    add("I1", "formal, shared knowledge representation", "OK" if all("@context" in s for s in sc) and LEDGER_SCHEMA.exists() else "MISSING",
        "JSON-LD context (Dublin Core Terms + PROV-O) in sidecars; Table Schema for the ledger")
    add("I2", "FAIR vocabularies for all fields", "WARN", "only uuid / schema_version / parent / title / creator / date / license are mapped; origin, generator, sources are plain JSON keys")
    pc = sum("uuid" in s["parent"] for s in sc)
    add("I3", "qualified references to the source (parent.uuid)", "OK" if pc == len(sc) and sc else "WARN",
        f"{pct(pc)} resolve to a sidecar uuid" + ("" if pc == len(sc) else "; the rest are identified by file name only (set PARENT_SIDECAR_UUIDS)"))
    seeds = {str(s["generator"]["seed"]) for s in sc}
    add("R1", "generator identity (repo and version)", "OK" if GENERATOR.get("repo") and GENERATOR.get("version") else "MISSING",
        f"repo={GENERATOR.get('repo')} version={GENERATOR.get('version')}")
    add("R1", "reproducibility: distinct seeds", "OK" if len(seeds) > 1 or len(sc) <= 1 else "WARN", f"{len(seeds)} distinct seed(s) over {len(sc)} images")
    add("R1.1", "data licence declared", "OK" if DATASET.get("license") else "MISSING", DATASET.get("license") or "set DATASET['license'] (the repo's GPL-3.0 covers code only)")
    add("R1.1", "rights of the source frames and of the people shown documented", "OK" if DATASET.get("rights_note") else "WARN", "" if DATASET.get("rights_note") else "DATASET['rights_note'] is empty")
    oc = sum("origin" in s for s in sc)
    mc = sum("manifest_idx" in s.get("origin", {}) for s in sc)
    hc = sum("frame_sha256" in s.get("origin", {}) for s in sc)
    add("R1.2", "provenance: source frame identified (collection, film, frame, portable path)", "OK" if oc == len(sc) and sc else "WARN", pct(oc))
    add("R1.2", "provenance: source frame described by the origin manifest", "OK" if mc == len(sc) and sc else "WARN", pct(mc))
    add("R1.2", "provenance: source frame hash (frame_sha256)", "OK" if hc == len(sc) and sc else "WARN", pct(hc) + ("" if SOURCE_FRAMES_DIR else "; set SOURCE_FRAMES_DIR to hash the frames"))
    add("R1.3", "community standards: JSON Schema validation, Data Package", "OK" if report["sidecars failing the schema"] == 0 and (OUT_ROOT / "datapackage.json").exists() else "WARN", "")
    return pd.DataFrame(R)

fair = fair_report()
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print(fair.to_string(index=False, max_colwidth=72))
print("\n" + fair["status"].value_counts().reindex(["OK", "WARN", "MISSING"], fill_value=0).to_string())

## 7. Trace an image back to its source

The steps of `LINEAGE.md`: find the ledger row, read `parent_uuid`, open the sidecar and follow `parent` and `origin`.

In [ ]:
def trace(output=None, manipulation_id=None):
    row = led[(led["manipulation_id"] == manipulation_id) | (led["output_path"].str.endswith(str(output)) if output else False)]
    if row.empty:
        return print("not in the ledger")
    r = row.iloc[-1]
    sc = sidecars[r["manipulation_id"]]
    print(f"ledger row   {r['uuid']}  method={r['method']}  sha256={r['sha256'][:12]}...")
    print(f"             input ({r['source_type']}): {r['source_path']}   parent_uuid={r['parent_uuid'] or '(not known)'}")
    print(f"sidecar      {sc['uuid']}  type={sc['manipulation_type']}  depth={sc['manipulation_depth']}")
    print(f"             parent: {sc['parent']}")
    if "origin" in sc:
        print(f"             origin: {sc['origin']}")
    print(f"             seed={sc['generator']['seed']}  prompt={sc['generator']['params']['prompt']!r}")

if len(led):
    trace(manipulation_id=led["manipulation_id"].iloc[0])